In [ ]:
from jointfm_client import bootstrap_notebook
bootstrap_notebook(add_src_root=True)

# CSV Forecast Workflow
Load the checked-in positive-float finance history and call the `jointfm-client forecast-csv` command for all future features and targets across the next 10 ordinal horizons.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

from jointfm_client import JointFMClient, plan_forecast_columns

HISTORY_PATH = Path('notebooks/history.csv')
OUTPUT_PATH = Path('notebooks/forecast.csv')
FEATURE_COLUMNS = ['equity_index_level', 'treasury_10y_yield', 'eur_usd_rate']
TARGET_COLUMNS = ['portfolio_nav', 'realized_volatility']
REQUESTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
INPUT_STEPS = 100
OUTPUT_HORIZONS = 10
EXPECTED_COLUMNS = FEATURE_COLUMNS + TARGET_COLUMNS
QUERY_TIMES = list(range(INPUT_STEPS, INPUT_STEPS + OUTPUT_HORIZONS))
QUERY_TIMES_ARGUMENT = ','.join(str(query_time) for query_time in QUERY_TIMES)

history = pd.read_csv(HISTORY_PATH, dtype=float)
if list(history.columns) != EXPECTED_COLUMNS:
    raise ValueError(f'Expected columns {EXPECTED_COLUMNS!r}, got {list(history.columns)!r}')
if len(history) != INPUT_STEPS:
    raise ValueError(f'Expected {INPUT_STEPS} history rows, got {len(history)}')
non_float_columns = [
    column_name for column_name, dtype in history.dtypes.items() if dtype.kind != 'f'
]
if non_float_columns:
    raise ValueError(f'Legacy JointFM model requires float columns: {non_float_columns!r}')
non_positive_columns = [
    column_name for column_name in history.columns if (history[column_name] <= 0).any()
]
if non_positive_columns:
    raise ValueError(f'Legacy JointFM model requires positive columns: {non_positive_columns!r}')

client = JointFMClient.from_env()
plan = plan_forecast_columns(
    health=client.health(cache=True),
    feature_columns=FEATURE_COLUMNS,
    target_columns=TARGET_COLUMNS,
    history_length=len(history),
    query_times_length=len(QUERY_TIMES),
)

target_column_arguments = []
for column_name in plan.target_columns:
    target_column_arguments.extend(['--target-column', column_name])
requested_column_arguments = []
for column_name in REQUESTED_COLUMNS:
    requested_column_arguments.extend(['--requested-column', column_name])

subprocess.run(
    [
        sys.executable,
        '-m',
        'jointfm_client.cli',
        'forecast-csv',
        str(HISTORY_PATH),
        str(OUTPUT_PATH),
        '--query-times',
        QUERY_TIMES_ARGUMENT,
        *target_column_arguments,
        *requested_column_arguments,
        '--seed',
        '7',
    ],
    capture_output=True,
    text=True,
    check=True,
)
forecast = pd.read_csv(OUTPUT_PATH)
expected_forecast_rows = OUTPUT_HORIZONS * len(REQUESTED_COLUMNS)
if len(forecast) != expected_forecast_rows:
    raise ValueError(f'Expected {expected_forecast_rows} forecast rows, got {len(forecast)}')
forecast